# Fine-tuning the helmet / head detector (YOLO11) on Google Colab

This notebook trains the detector that pre-labels images in the annotation app.

| Step | What happens |
|---|---|
| 1 | Check the GPU, mount Google Drive, install pinned libraries |
| 2 | Copy the dataset zip from Drive to the fast local disk and extract it |
| 3 | Fine-tune COCO-pretrained YOLO11 models (`n` and `s`) on our 3,800 training images |
| 4 | Evaluate each model on the **held-out test split** (600 images, never seen in training) |
| 5 | Log parameters, metrics and files to **MLflow** |
| 6 | Export the chosen model to **OpenVINO** for fast CPU inference in the backend |

**Before you run:** upload `hardhat-prepared.zip` (made by `backend/scripts/package_dataset.py`) to
`MyDrive/annotator/`, then choose *Runtime → Change runtime type → T4 GPU* and *Runtime → Run all*.

Everything that must survive a disconnect (training runs, MLflow database, exported model) is written
to `MyDrive/annotator/`. If Colab disconnects, run all cells again: training **resumes** from the last
saved epoch.

## 1. Setup

In [ ]:
# A GPU trains ~30x faster than a CPU. If this fails: Runtime -> Change runtime type -> T4 GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Pinned versions: the same notebook gives the same results next month (reproducibility).
# The backend must use the same ultralytics version to load the exported model.
!pip install -q "ultralytics==8.4.215" "mlflow==3.18.0" "openvino==2026.4.1"

In [ ]:
import csv
import json
import shutil
import time
import zipfile
from pathlib import Path

import mlflow
import ultralytics
from ultralytics import YOLO, settings

DRIVE_DIR = Path("/content/drive/MyDrive/annotator")
ZIP_ON_DRIVE = DRIVE_DIR / "hardhat-prepared.zip"
DATA_DIR = Path("/content/hardhat-prepared")  # local disk: much faster to read than Drive
RUNS_DIR = DRIVE_DIR / "runs"  # weights + training curves, kept on Drive
MLFLOW_DB = Path("/content/mlflow.db")  # SQLite needs file locking, which Drive does not support:
MLFLOW_DB_ON_DRIVE = DRIVE_DIR / "mlflow.db"  # work locally, copy to Drive after logging
CLASSES = ["helmet", "head"]

# --- Experiment settings (the report's "training setup" table comes from here) ---
MODELS = ["yolo11n.pt", "yolo11s.pt"]  # COCO-pretrained starting points: nano and small
EPOCHS = 60  # upper limit; early stopping usually ends sooner
PATIENCE = 15  # stop when validation mAP has not improved for 15 epochs
IMGSZ = 640  # images are ~416 px; upscaling to 640 helps with the many small heads
BATCH = 32  # fits a 16 GB T4 for both model sizes at 640 px
SEED = 42  # same seed as the dataset split

# Ultralytics has its own MLflow hook; we log explicitly instead, so it is clear what is recorded.
settings.update({"mlflow": False})
print("ultralytics", ultralytics.__version__, "| mlflow", mlflow.__version__)

## 2. Dataset

In [ ]:
if not ZIP_ON_DRIVE.exists():
    raise FileNotFoundError(f"Upload hardhat-prepared.zip to {ZIP_ON_DRIVE.parent} first")

if not (DATA_DIR / "data.yaml").exists():
    start = time.time()
    local_zip = Path("/content/hardhat-prepared.zip")
    shutil.copy(ZIP_ON_DRIVE, local_zip)  # one big copy is much faster than many small reads
    with zipfile.ZipFile(local_zip) as zf:
        zf.extractall(DATA_DIR)
    local_zip.unlink()
    print(f"Extracted in {time.time() - start:.0f} s")

DATA_YAML = DATA_DIR / "data.yaml"
# Always write an ABSOLUTE dataset path: Ultralytics resolves a relative `path:` against the
# current working directory (/content), not against the folder of data.yaml.
names = "".join(f"  {i}: {name}\n" for i, name in enumerate(CLASSES))
DATA_YAML.write_text(
    f"path: {DATA_DIR}\ntrain: train/images\nval: val/images\ntest: test/images\nnames:\n{names}"
)
for split in ("train", "val", "test"):
    n_images = len(list((DATA_DIR / split / "images").iterdir()))
    n_labels = len(list((DATA_DIR / split / "labels").iterdir()))
    print(f"{split:5s}: {n_images} images, {n_labels} label files")

## 3. Training

Fine-tuning = starting from weights already trained on COCO (80 everyday classes) instead of random
weights. The early layers already detect edges, shapes and heads, so 3,800 images are enough to learn
our two classes. Training from scratch would need far more data and time.

In [ ]:
def train(model_name: str) -> Path:
    # Train one model (or resume it after a disconnect) and return the path of best.pt.
    run_name = Path(model_name).stem  # e.g. "yolo11n"
    run_dir = RUNS_DIR / run_name
    best, last = run_dir / "weights" / "best.pt", run_dir / "weights" / "last.pt"

    if best.exists() and (run_dir / "done.txt").exists():
        print(f"{run_name}: already trained, skipping")
        return best
    if last.exists():
        print(f"{run_name}: resuming from {last}")
        YOLO(last).train(resume=True)
    else:
        YOLO(model_name).train(
            data=str(DATA_YAML),
            epochs=EPOCHS,
            patience=PATIENCE,
            imgsz=IMGSZ,
            batch=BATCH,
            seed=SEED,
            deterministic=True,
            # Pre-decoded images on local disk: fast loading with only 2 CPU cores, reproducible,
            # and no RAM build-up (cache="ram" ran out of memory when training a second model).
            cache="disk",
            workers=2,
            project=str(RUNS_DIR),
            name=run_name,
            exist_ok=True,
            plots=True,
        )
    (run_dir / "done.txt").write_text("finished\n")
    return best


best_weights = {Path(m).stem: train(m) for m in MODELS}
best_weights

## 4. Evaluation on the held-out test split

The validation split was used during training (to pick the best epoch), so it is slightly optimistic.
The **test split** was never touched: these are the numbers that go into the report.

* **mAP@0.5**: a detection counts as correct if IoU >= 0.5 with a ground-truth box.
* **mAP@0.5:0.95**: the COCO metric, averaged over IoU thresholds 0.5, 0.55, ..., 0.95 (rewards tight boxes).
* **Precision / recall**: at the confidence threshold that maximises F1.

In [ ]:
def evaluate(name: str, weights: Path) -> dict:
    m = YOLO(weights).val(
        data=str(DATA_YAML),
        split="test",
        imgsz=IMGSZ,
        batch=BATCH,
        plots=True,
        project=str(RUNS_DIR),
        name=f"{name}-test",
        exist_ok=True,
    )
    result = {
        "model": name,
        "test_images": 600,
        "map50": round(float(m.box.map50), 4),
        "map50_95": round(float(m.box.map), 4),
        "precision": round(float(m.box.mp), 4),
        "recall": round(float(m.box.mr), 4),
        "per_class": {
            cls: {"ap50": round(float(m.box.ap50[i]), 4), "ap50_95": round(float(m.box.ap[i]), 4)}
            for i, cls in enumerate(CLASSES)
        },
        "gpu_ms_per_image": round(m.speed["inference"], 2),
        "params_millions": round(sum(p.numel() for p in YOLO(weights).model.parameters()) / 1e6, 2),
    }
    print(json.dumps(result, indent=2))
    return result


test_results = {name: evaluate(name, w) for name, w in best_weights.items()}

## 5. Log to MLflow

MLflow keeps every experiment (settings, metrics, files) in one place, so results stay comparable
across runs. The database lives on Drive; files (curves, weights) go to `MyDrive/annotator/mlartifacts`.

In [ ]:
if MLFLOW_DB_ON_DRIVE.exists():
    shutil.copy(MLFLOW_DB_ON_DRIVE, MLFLOW_DB)  # continue the history of earlier sessions
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB}")
EXPERIMENT = "helmet-detector"
if mlflow.get_experiment_by_name(EXPERIMENT) is None:
    mlflow.create_experiment(EXPERIMENT, artifact_location=str(DRIVE_DIR / "mlartifacts"))
mlflow.set_experiment(EXPERIMENT)

for name, result in test_results.items():
    run_dir = RUNS_DIR / name
    with mlflow.start_run(run_name=name):
        mlflow.log_params(
            {
                "base_weights": f"{name}.pt (COCO-pretrained)",
                "epochs_max": EPOCHS,
                "patience": PATIENCE,
                "imgsz": IMGSZ,
                "batch": BATCH,
                "seed": SEED,
                "ultralytics": ultralytics.__version__,
            }
        )
        mlflow.log_metrics(
            {f"test_{k}": v for k, v in result.items() if isinstance(v, float)}
            | {f"test_ap50_{cls}": v["ap50"] for cls, v in result["per_class"].items()}
        )
        # Per-epoch curves, so MLflow can plot the learning curves.
        with open(run_dir / "results.csv") as f:
            for row in csv.DictReader(f):
                row = {k.strip(): v for k, v in row.items()}
                step = int(float(row["epoch"]))
                mlflow.log_metric("val_map50", float(row["metrics/mAP50(B)"]), step=step)
                mlflow.log_metric("val_map50_95", float(row["metrics/mAP50-95(B)"]), step=step)
                if "train/box_loss" in row:  # loss column names depend on the model type
                    mlflow.log_metric("train_box_loss", float(row["train/box_loss"]), step=step)
        for artifact in ("results.png", "confusion_matrix.png", "args.yaml", "weights/best.pt"):
            if (run_dir / artifact).exists():
                mlflow.log_artifact(str(run_dir / artifact))

shutil.copy(MLFLOW_DB, MLFLOW_DB_ON_DRIVE)
summary_path = DRIVE_DIR / "detector_metrics.json"
summary_path.write_text(json.dumps(test_results, indent=2))
print(f"Saved {summary_path} -> copy it to eval/results/ in the repo")

## 6. Export the chosen model to OpenVINO

The backend runs on a laptop CPU (no GPU), so the model is converted to **OpenVINO**, Intel's inference
runtime, which is typically several times faster than plain PyTorch on Intel CPUs.

**Which model?** Pick the one with the best accuracy that is still fast on CPU. `yolo11n` is ~3x
smaller than `yolo11s`; choose `s` only if it is clearly more accurate (e.g. +0.03 mAP@0.5:0.95).
The CPU speed is measured on the laptop in Phase 4.

In [ ]:
CHOSEN = max(test_results, key=lambda n: test_results[n]["map50_95"])  # override by hand if needed
print("Exporting", CHOSEN)

export_path = Path(YOLO(best_weights[CHOSEN]).export(format="openvino", imgsz=IMGSZ, dynamic=False))
target = DRIVE_DIR / f"{CHOSEN}_helmet_openvino_model"
shutil.copytree(export_path, target, dirs_exist_ok=True)
shutil.copy(best_weights[CHOSEN], DRIVE_DIR / f"{CHOSEN}_helmet.pt")
print("Saved to Drive:", target, "and", DRIVE_DIR / f"{CHOSEN}_helmet.pt")

## 7. Look at the results

Open `MyDrive/annotator/runs/<model>/` for the training curves (`results.png`), confusion matrix and
example predictions (`val_batch*_pred.jpg`). Look for **failure cases** for the report: small/far heads,
occlusion, unusual helmet colours, hats that are not helmets.

To browse MLflow on your laptop: download `mlflow.db` from Drive, then
`uvx mlflow ui --backend-store-uri sqlite:///mlflow.db` and open http://localhost:5000.

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(RUNS_DIR / CHOSEN / "results.png"), width=900))
display(
    Image(filename=str(RUNS_DIR / f"{CHOSEN}-test" / "confusion_matrix_normalized.png"), width=600)
)